# Section 2: Read any shipping document with Document AI

Goal: turn Harbor Freight's PDFs (clean and scanned) into typed fields Dock can trust,
and flag the documents that are wrong before anyone acts on them.

## Step 1. Look at the documents

In [1]:
from pathlib import Path
import pymupdf  # opens PDFs

docs = sorted(Path("dock/documents").glob("*.pdf"))
for path in docs:
    text = pymupdf.open(path)[0].get_text()
    print(f"{path.name:30} text layer: {len(text):4} characters")

bill_of_lading_01.pdf          text layer:  550 characters
bill_of_lading_02.pdf          text layer:  552 characters
bill_of_lading_03_scan.pdf     text layer:    0 characters
bill_of_lading_04_scan.pdf     text layer:    0 characters
invoice_01_clean.pdf           text layer:  659 characters
invoice_02_clean.pdf           text layer:  662 characters
invoice_03_eu_numbers.pdf      text layer:  628 characters
invoice_04_scan.pdf            text layer:    0 characters
invoice_05_scan_stain.pdf      text layer:    0 characters
invoice_06_scan_eu.pdf         text layer:    0 characters
invoice_07_scan_mismatch.pdf   text layer:    0 characters


## Step 2. Look at the database

In [2]:
import sqlite3

db = sqlite3.connect("dock/dock.sqlite")
tables = db.execute("SELECT name FROM sqlite_master WHERE type='table'")
for (name,) in tables.fetchall():
    count = db.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
    print(f"{name:16} {count:3} rows")

customers          8 rows
shipments         40 rows
invoices          40 rows
invoice_lines     80 rows
tracking_events   75 rows


In [3]:
def show(sql, *args):
    cur = db.execute(sql, args)
    names = [col[0] for col in cur.description]
    for row in cur.fetchall():
        for name, value in zip(names, row):
            print(f"  {name:16} {value}")
        print()

show("SELECT * FROM shipments WHERE id = ?", "HF-26040")

  id               HF-26040
  shipper_id       C04
  consignee_id     C05
  origin_port      JPYOK Yokohama
  destination_port GBFXT Felixstowe
  container_no     HRBU4153430
  incoterm         DAP
  status           customs hold
  etd              2026-11-18
  eta              2026-12-23



In [4]:
show("SELECT * FROM invoices WHERE shipment_id = ?", "HF-26040")
show("SELECT event, note FROM tracking_events WHERE shipment_id = ?", "HF-26040")

  invoice_no       INV-2026-3139
  shipment_id      HF-26040
  issue_date       2026-11-18
  currency         USD
  subtotal         3630.0
  freight          217.8
  insurance        14.52
  total            3862.32

  event            booking confirmed
  note             

  event            gate in
  note             

  event            customs hold
  note             Commercial invoice total does not match declared value; awaiting corrected invoice.



## Step 3. Connect to Mistral

In [5]:
import os
from dotenv import load_dotenv
from mistralai.client import Mistral
from mistralai.client.utils import BackoffStrategy, RetryConfig

load_dotenv()
retry = RetryConfig("backoff", BackoffStrategy(1000, 20000, 2.0, 120000),
                    retry_connection_errors=True)
client = Mistral(api_key=os.environ["MISTRAL_API_KEY"], retry_config=retry,
                 timeout_ms=120000)
print("key loaded:", len(os.environ["MISTRAL_API_KEY"]) > 0)

key loaded: True


## Step 4. OCR one scanned invoice

In [6]:
import base64

def as_data_url(path):
    data = base64.b64encode(Path(path).read_bytes()).decode()
    return f"data:application/pdf;base64,{data}"

scan = "dock/documents/invoice_04_scan.pdf"
url = as_data_url(scan)
result = client.ocr.process(
    model="mistral-ocr-latest",
    document={"type": "document_url", "document_url": url},
    confidence_scores_granularity="word",
)
print(result.usage_info)

pages_processed=1 doc_size_bytes=234935


In [7]:
page = result.pages[0]
print(page.markdown[:950])

# COMMERCIAL INVOICE

Seller / Exporter

Kanto Precision Parts K.K.

Yokohama, Japan

shipping@kanto-pp.example.com

Buyer / Importer

Brightwater Textiles Ltd.

Felixstowe, United Kingdom

freight@brightwater.example.com

Invoice No.: INV-2026-3115

Date: 2026-10-01

Shipment ref.: HF-26016

Container: HRBU4120550

Incoterm: CIF Felixstowe

|  Description | HS code | Qty | Unit price | Amount  |
| --- | --- | --- | --- | --- |
|  Precision ball bearing 6204-2RS | 8482.10 | 40 pcs | 1.85 | 74.00  |
|  CNC machined aluminium bracket | 7616.99 | 1500 pcs | 4.20 | 6,300.00  |
|   |  |  | Subtotal | 6,374.00  |
|   |  |  | Freight | 382.44  |
|   |  |  | Insurance | 25.50  |
|   |  |  | **TOTAL USD** | **6,781.94**  |

We certify that this invoice is true and correct. Harbor Freight Co. is the forwarder.


## Step 5. How sure was it?

In [8]:
scores = page.confidence_scores
print("average:", round(scores.average_page_confidence_score, 4))
print("minimum:", round(scores.minimum_page_confidence_score, 4))
words = sorted(scores.word_confidence_scores, key=lambda w: w.confidence)
for w in words[:5]:
    print(f"  {w.text!r:18} {w.confidence:.3f}")

average: 0.9982
minimum: 0.8962
  'Incoterm:'        0.896
  'Description'      0.993
  'HS'               0.993
  ' Ltd.'            0.995
  ' pcs'             0.998


## Step 6. Ask for fields: a first schema

In [9]:
from pydantic import BaseModel
from mistralai.extra import response_format_from_pydantic_model

class Line(BaseModel):
    description: str
    hs_code: str
    quantity: int
    unit_price: float
    amount: float

class InvoiceV1(BaseModel):
    invoice_no: str
    shipment_id: str
    container_no: str
    incoterm: str
    seller: str
    buyer: str
    lines: list[Line]
    subtotal: float
    freight: float
    insurance: float
    total: float

In [10]:
def extract(path, schema):
    url = as_data_url(path)
    fmt = response_format_from_pydantic_model(schema)
    response = client.ocr.process(
        model="mistral-ocr-latest",
        document={"type": "document_url", "document_url": url},
        document_annotation_format=fmt,
    )
    return schema.model_validate_json(response.document_annotation)

v1 = extract(scan, InvoiceV1)
print(v1.invoice_no, "|", v1.incoterm, "|", v1.total)
print(repr(v1.seller))

INV-2026-3115 | CIF Felixstowe | 6781.94
'Kanto Precision Parts K.K., Yokohama, Japan'


In [11]:
def find_customer(name):
    row = db.execute("SELECT id FROM customers WHERE company = ?", (name,))
    return row.fetchone()

print("seller in database?", find_customer(v1.seller))

seller in database? None


## Step 7. A better schema: the descriptions are the prompt

In [12]:
from typing import Literal
from pydantic import Field

class Invoice(BaseModel):
    invoice_no: str
    shipment_id: str = Field(description="Harbor Freight ref, like HF-26001")
    container_no: str = Field(description="11 characters, like HRBU4100003")
    incoterm: Literal["EXW", "FOB", "CIF", "DAP"]
    incoterm_place: str = Field(description="Place after the incoterm")
    seller: str = Field(description="Company name only, no address")
    buyer: str = Field(description="Company name only, no address")
    lines: list[Line]
    subtotal: float
    freight: float
    insurance: float
    total: float = Field(description="Grand total exactly as printed")

In [13]:
invoices = [p for p in docs if p.name.startswith("invoice")]
extracted = {}
for path in invoices:
    extracted[path.name] = extract(path, Invoice)
    inv = extracted[path.name]
    ok = find_customer(inv.seller) is not None
    print(f"{path.name:30} {inv.incoterm} {inv.total:>10,.2f}  seller found: {ok}")

invoice_01_clean.pdf           CIF  34,218.24  seller found: True


invoice_02_clean.pdf           FOB  14,970.00  seller found: True


invoice_03_eu_numbers.pdf      FOB  19,125.00  seller found: True


invoice_04_scan.pdf            CIF   6,781.94  seller found: True


invoice_05_scan_stain.pdf      EXW  29,920.00  seller found: True


invoice_06_scan_eu.pdf         DAP  23,365.44  seller found: True


invoice_07_scan_mismatch.pdf   DAP   8,362.32  seller found: True


## Step 8. Check what we read

In [14]:
def check_digit(code10):
    values, n = {}, 10
    for ch in "ABCDEFGHIJKLMNOPQRSTUVWXYZ":
        if n % 11 == 0:
            n += 1
        values[ch] = n
        n += 1
    digits = [values[c] if c.isalpha() else int(c) for c in code10]
    return sum(d * 2 ** i for i, d in enumerate(digits)) % 11 % 10

for box in ["HRBU4100003", "HRBU4100903"]:
    print(box, "computed:", check_digit(box[:10]), "printed:", box[10])

HRBU4100003 computed: 3 printed: 3
HRBU4100903 computed: 8 printed: 3


In [15]:
def problems(inv):
    found = []
    lines_sum = round(sum(l.amount for l in inv.lines), 2)
    if abs(lines_sum - inv.subtotal) > 0.01:
        found.append(f"lines add up to {lines_sum:,.2f}, not {inv.subtotal:,.2f}")
    expected = round(inv.subtotal + inv.freight + inv.insurance, 2)
    if abs(expected - inv.total) > 0.01:
        found.append(f"total should be {expected:,.2f}, printed {inv.total:,.2f}")
    box = inv.container_no
    if len(box) != 11 or check_digit(box[:10]) != int(box[10]):
        found.append(f"container {box} fails the check digit")
    return found

for name, inv in extracted.items():
    print(f"{name:30} {problems(inv) or 'OK'}")

invoice_01_clean.pdf           OK
invoice_02_clean.pdf           OK
invoice_03_eu_numbers.pdf      OK
invoice_04_scan.pdf            OK
invoice_05_scan_stain.pdf      OK
invoice_06_scan_eu.pdf         OK
invoice_07_scan_mismatch.pdf   ['total should be 3,862.32, printed 8,362.32']


In [16]:
def against_database(inv):
    row = db.execute("SELECT total FROM invoices WHERE invoice_no = ?",
                     (inv.invoice_no,)).fetchone()
    if row is None:
        return "invoice not in database"
    if abs(row[0] - inv.total) > 0.01:
        return f"database says {row[0]:,.2f}"
    return "matches database"

for name, inv in extracted.items():
    print(f"{name:30} {against_database(inv)}")

invoice_01_clean.pdf           matches database
invoice_02_clean.pdf           matches database
invoice_03_eu_numbers.pdf      matches database
invoice_04_scan.pdf            matches database
invoice_05_scan_stain.pdf      matches database
invoice_06_scan_eu.pdf         matches database
invoice_07_scan_mismatch.pdf   database says 3,862.32


## Step 9. Bills of lading: same pattern, new schema

In [17]:
class BillOfLading(BaseModel):
    bl_no: str
    shipment_id: str = Field(description="Harbor Freight ref, like HF-26001")
    shipper: str = Field(description="Company name only, no address")
    consignee: str = Field(description="Company name only, no address")
    container_no: str = Field(description="11 characters, like HRBU4100003")
    packages: int
    gross_weight_kg: float

for path in [p for p in docs if p.name.startswith("bill")]:
    bl = extract(path, BillOfLading)
    box = bl.container_no
    digit_ok = check_digit(box[:10]) == int(box[10])
    print(f"{path.name:28} {bl.shipment_id} {box} check digit ok: {digit_ok}")

bill_of_lading_01.pdf        HF-26001 HRBU4100003 check digit ok: True


bill_of_lading_02.pdf        HF-26005 HRBU4105480 check digit ok: True


bill_of_lading_03_scan.pdf   HF-26020 HRBU4126034 check digit ok: True


bill_of_lading_04_scan.pdf   HF-26038 HRBU4153690 check digit ok: False


## Step 10. Score it against the answer key

In [18]:
import json

truth = json.load(open("dock/documents/truth.json"))
fields = ["invoice_no", "shipment_id", "container_no", "incoterm",
          "seller", "buyer", "subtotal", "freight", "insurance", "total"]
correct = 0
for name, inv in extracted.items():
    wrong = [f for f in fields if getattr(inv, f) != truth[name][f]]
    flagged = bool(problems(inv)) == bool(truth[name]["flags"])
    correct += not wrong and flagged
    print(f"{name:30} wrong fields: {wrong or '-'}  flag right: {flagged}")
print(f"\nDock Scorecard, documents: {correct}/{len(extracted)}")

invoice_01_clean.pdf           wrong fields: -  flag right: True
invoice_02_clean.pdf           wrong fields: -  flag right: True
invoice_03_eu_numbers.pdf      wrong fields: -  flag right: True
invoice_04_scan.pdf            wrong fields: -  flag right: True
invoice_05_scan_stain.pdf      wrong fields: -  flag right: True
invoice_06_scan_eu.pdf         wrong fields: -  flag right: True
invoice_07_scan_mismatch.pdf   wrong fields: -  flag right: True

Dock Scorecard, documents: 7/7
